# The Hyper-Contextual Recruiter
## Implementing the Converged Database (Vector + Spatial + Graph)

**Copyright 2026, Denis Rothman**

**Prerequisites:** This notebook assumes you have successfully run the **Data Ingestion** notebooks from Chapter 3 (`1_DBA_Oracle_Management_V2.ipynb` and `2_Data_Ingestion_Oracle_DB.ipynb`). The database must already contain the `CANDIDATE_POOL` table populated with candidate data.

### Overview
In previous chapters, we used **Vectors** to find candidates with the right skills. Now, we will upgrade the system to answer real-world questions like:
1.  **Location (Spatial):** *"Is this candidate within a 30-minute commute?"*
2.  **Trust (Graph):** *"Is this candidate referred by a trusted employee?"*

We will demonstrate the power of the **Converged Database** by executing a single SQL query that filters by **Meaning + Money + Location + Relationships** simultaneously.

**Strategy:** To avoid modifying the educational data from Chapter 3, this notebook will create a **Shadow Copy** of the candidate table (`CANDIDATE_POOL_GEO`) to add spatial and graph features safely.

# 1. Environment Setup

In [1]:
# 1.1 Install the Oracle Database Driver
!pip install oracledb==3.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.6 MB/s eta 0:00:00


In [2]:
# 1.2 Secure Database Connection
import oracledb
from google.colab import drive

# Mount Google Drive to access the Oracle Wallet and Credentials
drive.mount('/content/drive')

# Configuration Paths
wallet_path = "/content/drive/MyDrive/files/oracle"
creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"

def read_key_value_file(path):
    """Helper to read the secure credentials file."""
    creds = {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if "=" in line and not line.startswith("#"):
                    k, v = line.strip().split("=", 1)
                    creds[k.strip()] = v.strip()
    except FileNotFoundError:
        print("⚠️ Credentials file not found. Please check your Drive paths.")
    return creds

creds = read_key_value_file(creds_path)

# Establish Connection
try:
    connection = oracledb.connect(
        user=creds.get("user"),
        password=creds.get("password"),
        dsn=creds.get("dsn", "mqyv1gnzq40yxs41_high"),
        config_dir=wallet_path,
        wallet_location=wallet_path,
        wallet_password=creds.get("wallet_password")
    )
    cursor = connection.cursor()
    print("✅ Connected to Oracle 23ai successfully.")
except Exception as e:
    print(f"❌ Connection failed: {e}")

Mounted at /content/drive
✅ Connected to Oracle 23ai successfully.


In [3]:
# 1.3 Setup OpenAI (for Embedding & Generation)
!pip install openai==2.14.0

import os
from openai import OpenAI
from google.colab import userdata

try:
    api_key = userdata.get("API_KEY")
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI()
    print("✅ OpenAI API initialized.")
except Exception as e:
    print(f"⚠️ API Key Warning: {e}")

# Helper function for embeddings
def get_embedding(text):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

✅ OpenAI API initialized.


# 2. The Spatial Dimension: Geolocation

To enable spatial features without modifying the original `CANDIDATE_POOL` (from Chapter 3), we will create a dedicated table: `CANDIDATE_POOL_GEO`.

1.  **Clone Data:** Copy existing candidates into the new table.
2.  **Add Geometry:** Add an `SDO_GEOMETRY` column.
3.  **Simulate Locations:** Place candidates in San Francisco (HQ) or New York.


In [4]:
print("--- 🌍 Initializing Spatial Data ---\n")

# 1. Clear previous data to ensure idempotency (optional but safe)
cursor.execute("TRUNCATE TABLE candidate_pool_geo")

# 2. Populate from Source (Chapter 3 Data)
# Note: We explicitly select columns to match the new schema structure
print("⏳ Copying data from Chapter 3 Source...")
cursor.execute("""
    INSERT INTO candidate_pool_geo (
        candidate_id, full_name, summary, skills, years_experience, salary_expectation, resume_vector
    )
    SELECT
        candidate_id, full_name, summary, skills, years_experience, salary_expectation, resume_vector
    FROM candidate_pool
""")
print(f"✅ Data copied. Rows inserted: {cursor.rowcount}")

# 3. Update Spatial Locations (Update existing rows)
# Group A: San Francisco Downtown
cursor.execute("""
    UPDATE candidate_pool_geo
    SET home_location = SDO_GEOMETRY(2001, 4326, SDO_POINT_TYPE(-122.4194, 37.7749, NULL), NULL, NULL)
    WHERE full_name IN ('Alex V.', 'Casey M.')
""")

# Group B: SF Suburbs
cursor.execute("""
    UPDATE candidate_pool_geo
    SET home_location = SDO_GEOMETRY(2001, 4326, SDO_POINT_TYPE(-122.25, 37.80, NULL), NULL, NULL)
    WHERE full_name LIKE 'Quinn%'
""")

# Group C: New York
cursor.execute("""
    UPDATE candidate_pool_geo
    SET home_location = SDO_GEOMETRY(2001, 4326, SDO_POINT_TYPE(-74.0060, 40.7128, NULL), NULL, NULL)
    WHERE full_name LIKE 'Jordan%'
""")

connection.commit()
print("✅ Candidate locations updated in 'candidate_pool_geo'.")

--- 🌍 Initializing Spatial Data ---

⏳ Copying data from Chapter 3 Source...
✅ Data copied. Rows inserted: 5
✅ Candidate locations updated in 'candidate_pool_geo'.


# 3. The Social Dimension: SQL Property Graphs

To find candidates referred by trusted employees, we need to model relationships. We will use **SQL Property Graphs**, a native feature of Oracle 23ai.

We define three components:
1.  **Vertex Table (Employees):** The referrers.
2.  **Edge Table (Referrals):** The link between an Employee and a Candidate in `CANDIDATE_POOL_GEO`.
3.  **Property Graph Object:** A virtual view that lets us traverse these tables using graph syntax.


In [6]:
print("--- 🕸️ Initializing Graph Data ---\n")

# 1. Reset Data (Truncate tables)
cursor.execute("TRUNCATE TABLE referrals") # Child table first (DDL: Auto-commits)

# Fix: We must commit the DELETE before inserting to release row locks (Prevents ORA-12860)
cursor.execute("DELETE FROM employees")    # Parent table second (DML)
connection.commit()                        # <--- ADD THIS LINE

# 2. Insert Employees
cursor.execute("INSERT INTO employees VALUES ('EMP_001', 'Alice Stark', 'Principal Architect')")
print("✅ Employees populated.")

# 3. Insert Referrals
cursor.execute("INSERT INTO referrals VALUES ('EMP_001', 'CAND_005', 'Former Colleague')")
cursor.execute("INSERT INTO referrals VALUES ('EMP_001', 'CAND_002', 'University Friend')")
print("✅ Referrals populated.")

connection.commit()

# 4. Create Property Graph (Idempotent Drop/Create)
try:
    cursor.execute("DROP PROPERTY GRAPH recruitment_graph")
except: pass

print("⏳ Compiling Property Graph object...")
cursor.execute("""
    CREATE PROPERTY GRAPH recruitment_graph
    VERTEX TABLES (
        employees KEY (emp_id) LABEL employee PROPERTIES (name),
        candidate_pool_geo KEY (candidate_id) LABEL candidate PROPERTIES (full_name, candidate_id)
    )
    EDGE TABLES (
        referrals
            KEY (referrer_id, candidate_id)
            SOURCE KEY (referrer_id) REFERENCES employees (emp_id)
            DESTINATION KEY (candidate_id) REFERENCES candidate_pool_geo (candidate_id)
            LABEL referred_by PROPERTIES (relationship)
    )
""")
print("✅ Property Graph 'RECRUITMENT_GRAPH' created successfully.")

--- 🕸️ Initializing Graph Data ---

✅ Employees populated.
✅ Referrals populated.
⏳ Compiling Property Graph object...
✅ Property Graph 'RECRUITMENT_GRAPH' created successfully.


# 4. The Hyper-Query: Vector + Spatial + Graph

This is the core of the Converged Database. We execute **one single SQL statement** that combines three distinct data engines.

**The Query Logic:**
1.  **Vector Engine:** Calculate similarity between "Python Backend Developer" and the candidate's resume.
2.  **Spatial Engine:** Filter candidates in `CANDIDATE_POOL_GEO` who live within `max_dist` miles.
3.  **Graph Engine:** Filter candidates who have a valid referral path from `referrer_name`.


In [7]:
# 1. Define the Search Criteria
query_text = "Python Backend Developer"
referrer_name = "Alice Stark"
max_distance_miles = 25
sf_office_lat = 37.7749
sf_office_lon = -122.4194

print(f"🔎 Searching for: '{query_text}'")
print(f"📍 Spatial Filter: Within {max_distance_miles} miles of SF Office")
print(f"🤝 Graph Filter:   Must be referred by '{referrer_name}'")

# 2. Vectorize
query_vec = get_embedding(query_text)

# 3. Execute the Converged Query
# CHANGE: FROM 'candidate_pool_geo'
sql_hyper = """
    SELECT
        c.full_name,
        c.salary_expectation,
        c.summary,
        -- Spatial Engine
        ROUND(SDO_GEOM.SDO_DISTANCE(
            c.home_location,
            SDO_GEOMETRY(2001, 4326, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL),
            0.005, 'unit=MILE'
        ), 1) as dist_miles,
        -- Vector Engine
        ROUND(VECTOR_DISTANCE(c.resume_vector, :vec, DOT), 4) as semantic_score
    FROM candidate_pool_geo c  -- <--- UPDATED TABLE

    -- Graph Engine (Label 'candidate' now points to candidate_pool_geo)
    INNER JOIN GRAPH_TABLE(recruitment_graph
        MATCH (e IS employee WHERE e.name = :ref_name)
        -[r IS referred_by]->
        (c_target IS candidate)
        COLUMNS (c_target.candidate_id AS g_id)
    ) gt ON c.candidate_id = gt.g_id

    WHERE
        -- Spatial Engine
        SDO_GEOM.SDO_DISTANCE(
            c.home_location,
            SDO_GEOMETRY(2001, 4326, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL),
            0.005, 'unit=MILE'
        ) < :max_dist

    ORDER BY semantic_score DESC
"""

cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
cursor.execute(sql_hyper, {
    "lon": sf_office_lon,
    "lat": sf_office_lat,
    "vec": query_vec,
    "max_dist": max_distance_miles,
    "ref_name": referrer_name
})

results = cursor.fetchall()

print("\n--- 🏆 Hyper-Query Results ---")
if not results:
    print("No candidates found matching all criteria.")
else:
    for r in results:
        name, salary, summary_lob, miles, score = r
        print(f"✅ {name}")
        print(f"   - Distance: {miles} miles from HQ")
        print(f"   - Semantic Score: {score}")
        print(f"   - Status: Verified Referral")

🔎 Searching for: 'Python Backend Developer'
📍 Spatial Filter: Within 25 miles of SF Office
🤝 Graph Filter:   Must be referred by 'Alice Stark'

--- 🏆 Hyper-Query Results ---
✅ Quinn R.
   - Distance: 9.4 miles from HQ
   - Semantic Score: -0.4134
   - Status: Verified Referral


# 5. The AI Recruiter (Generation)

We wrap the entire system in a Python function, `generate_hyper_recommendation`.
This function retrieves the data and uses GPT-4/5 to explain *why* the candidate is the best fit, specifically referencing the trust (referral) and presence (location) factors.

In [8]:
def generate_hyper_recommendation(user_query, referrer, max_dist_miles):
    """
    1. Retrieval: Runs the Oracle Spatial + Graph + Vector query.
    2. Generation: Sends the results to GPT to explain the match.
    """
    print(f"🤖 AI AGENT: Searching for '{user_query}'...")
    print(f"   Constraints: Within {max_dist_miles} miles of SF, Referred by '{referrer}'")

    # 1. Embed the user query
    query_vec = get_embedding(user_query)

    # 2. Execute Hyper-Query (Same SQL as above, ensuring we target candidate_pool_geo)
    cursor.setinputsizes(vec=oracledb.DB_TYPE_VECTOR)
    cursor.execute(sql_hyper, {
        "lon": sf_office_lon,
        "lat": sf_office_lat,
        "vec": query_vec,
        "max_dist": max_dist_miles,
        "ref_name": referrer
    })
    candidates = cursor.fetchall()

    if not candidates:
        return "⚠️ No candidates found. Try relaxing the distance or referral constraints."

    # 3. Format Context for LLM
    context_text = ""
    for c in candidates:
        name, salary, summary_lob, miles, score = c
        summary = summary_lob.read() if hasattr(summary_lob, 'read') else str(summary_lob)

        context_text += f"""
        --- CANDIDATE ---
        Name: {name}
        Salary Expectation: ${salary:,}
        Location: {miles} miles from office (Commutable)
        Referral: Verified ({referrer})
        Vector Match Score: {score}
        Resume Summary: {summary}
        -----------------
        """

    # 4. Generate Prompt
    system_prompt = "You are a Hyper-Contextual Recruiter. You value TRUST (Referrals) and PRESENCE (Location) as much as SKILL."
    user_prompt = f"""
    USER REQUEST: "{user_query}"

    CANDIDATES FOUND (Filtered by Graph & Spatial DB):
    {context_text}

    TASK:
    1. Recommend the best candidate.
    2. Explicitly mention why their location ({max_dist_miles} mile radius) and referral ({referrer}) make them a safer hire than an unknown candidate.
    """

    # 5. Call LLM
    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


# --- LIVE SIMULATION ---
rec_output = generate_hyper_recommendation(
    user_query="Senior Python Engineer",
    referrer="Alice Stark",
    max_dist_miles=25
)

print("\n" + "="*60)
print("📢 AI RECOMMENDATION")
print("="*60)
print(rec_output)

🤖 AI AGENT: Searching for 'Senior Python Engineer'...
   Constraints: Within 25 miles of SF, Referred by 'Alice Stark'

📢 AI RECOMMENDATION
## 1) Best candidate recommendation
**Recommend: Quinn R.** for the **Senior Python Engineer** role (with a quick calibration on current hands-on depth given their recent Engineering Manager focus).

Why Quinn is the best available from the provided set:
- **Strong Python background** plus senior-level maturity in communication, mentorship, and aligning engineering work to business outcomes—often a net positive for a senior IC role that influences others.
- **Culture and team impact**: Their emphasis on mentorship and goal alignment typically translates into higher leverage execution and better cross-functional delivery.

Caveat to validate in interview:
- Their resume signals a shift toward management; confirm recent **hands-on Python contribution** (system design + coding in the last 6–12 months) to ensure fit for a senior engineer seat.

## 2) W

# 6. Data Verification

Finally, we verify that our Shadow Table (`CANDIDATE_POOL_GEO`) was created correctly and that we did **not** modify the original table.

In [9]:
# Diagnostic Cell: Inspect the NEW Spatial Table
print("--- Current Columns in CANDIDATE_POOL_GEO ---")

try:
    # Query the Oracle Data Dictionary for our NEW shadow table
    cursor.execute("""
        SELECT column_name, data_type
        FROM user_tab_columns
        WHERE table_name = 'CANDIDATE_POOL_GEO'
        ORDER BY column_id
    """)

    columns = cursor.fetchall()
    spatial_found = False

    for col in columns:
        col_name, col_type = col
        print(f" 🔹 {col_name}: {col_type}")
        if col_name == 'HOME_LOCATION':
            spatial_found = True

    print("-" * 40)
    if not spatial_found:
        print("⚠️ CRITICAL ISSUE: The 'HOME_LOCATION' column is missing from CANDIDATE_POOL_GEO.")
    else:
        print("✅ Success: 'HOME_LOCATION' column found in the shadow table.")
        print("   (The original Chapter 3 table 'CANDIDATE_POOL' remains untouched)")

except Exception as e:
    print(f"❌ Error inspecting table: {e}")

--- Current Columns in CANDIDATE_POOL_GEO ---
 🔹 CANDIDATE_ID: VARCHAR2
 🔹 FULL_NAME: VARCHAR2
 🔹 SUMMARY: CLOB
 🔹 SKILLS: VARCHAR2
 🔹 YEARS_EXPERIENCE: NUMBER
 🔹 SALARY_EXPECTATION: NUMBER
 🔹 RESUME_VECTOR: VECTOR
 🔹 HOME_LOCATION: SDO_GEOMETRY
----------------------------------------
✅ Success: 'HOME_LOCATION' column found in the shadow table.
   (The original Chapter 3 table 'CANDIDATE_POOL' remains untouched)
